In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import tensorflow as tf
import yaml

sys.path.append(os.path.abspath("../"))

from util import camera as u_camera
from util import dataset as u_dataset
from util import dataset_io as u_dataset_io
from util import labels as u_labels
from util import metrics as u_metrics

In [ ]:
model_timestamp = "20260206-120402"

path_to_selected = Path("..", "..", "data", "selected")
test_groundtruth_path = Path(path_to_selected, "test_groundtruth.json")
test_bhuman_path = Path(path_to_selected, "test_b-human_predictions.json")

test_model_intersections_path = Path(path_to_selected, model_timestamp, "intersections.json")
test_model_ball_path = Path(path_to_selected, model_timestamp, "ball.json")
test_model_penaltymark_path = Path(path_to_selected, model_timestamp, "penaltyMark.json")

test_groundtruth = None
test_bhuman = None
test_intersections_model = None
test_ball_model = None
test_penaltymark_model = None

with open(test_groundtruth_path) as f:
    test_groundtruth = json.load(f)
with open(test_bhuman_path) as f:
    test_bhuman = json.load(f)
with open(test_model_intersections_path) as f:
    test_intersections_model = json.load(f)
with open(test_model_ball_path) as f:
    test_ball_model = json.load(f)
with open(test_model_penaltymark_path) as f:
    test_penaltymark_model = json.load(f)

# test_ball_model = test_ball_model[:20]
# test_bhuman = test_bhuman[:20]
# test_groundtruth = test_groundtruth[:20]

assert len(test_groundtruth) == len(test_bhuman)
assert len(test_bhuman) == len(test_intersections_model)
assert len(test_bhuman) == len(test_ball_model)
assert len(test_bhuman) == len(test_penaltymark_model)

save_path_for_matches = Path(path_to_selected, model_timestamp, "matches")
os.makedirs(save_path_for_matches, exist_ok=True)

In [ ]:
config_path = f"../../logs/fit/{model_timestamp}/config.yaml"

with open(config_path) as f:
    config = yaml.safe_load(f)

## Intersections


In [ ]:
def extract_coordinates(intersections):
    # Convert list of dicts to list of lists: [[x1, y1], [x2, y2], ...]
    return [[coord["x"], coord["y"]] for coord in intersections]

In [ ]:
def filter_coords_by_distance(coords_tensor, camera, intrinsics, max_distance: float):
    if tf.shape(coords_tensor)[0] > 0:
        if len(tf.shape(coords_tensor)) == 3:
            gt_coords_tensor = coords_tensor[:, 1]  # (N, 2)
            pred_coords_tensor = coords_tensor[:, 0]  # (N, 2)
        else:
            gt_coords_tensor = coords_tensor  # (N, 2)
            pred_coords_tensor = None

        img_to_wlrd = u_camera.image_to_world(camera, intrinsics, gt_coords_tensor)  # (N, 3)
        valid_distances = ~tf.reduce_all(img_to_wlrd == -1.0, axis=-1)  # (N, )

        distances = tf.linalg.norm(img_to_wlrd, axis=-1, keepdims=True)  # (N, )

        mask = tf.squeeze(distances <= max_distance, axis=-1) & valid_distances  # (N, )

        filtered_gt_tensor = tf.boolean_mask(gt_coords_tensor, mask)  # (M, 2)

        if pred_coords_tensor is not None:
            filtered_pred_tensor = tf.boolean_mask(pred_coords_tensor, mask)  # (M, 2)
            assert tf.reduce_all(tf.shape(filtered_gt_tensor) == tf.shape(filtered_pred_tensor))
            return tf.stack([filtered_pred_tensor, filtered_gt_tensor], axis=1)  # (M, 2, 2)
        else:
            return filtered_gt_tensor  # (M, 2)
    else:
        return tf.zeros((0, 2, 2), tf.float32)  # (0, 2, 2)

In [ ]:
def process_object_metrics(
    preds: dict[tf.Tensor],
    gt_frame: dict[tf.Tensor],
    object_name: str,
    threshold: float,
    intersection_type: str = None,
    ball_status_only_seen: bool = None,
) -> dict:
    # Extract coordinates for the given object type
    if object_name == u_dataset.CategoryNames.INTERSECTIONS.value:
        pred_coords = extract_coordinates(preds[object_name][intersection_type])
        gt_coords = extract_coordinates(gt_frame[object_name][intersection_type])
    elif object_name in [
        u_dataset.CategoryNames.BALL.value,
        u_dataset.CategoryNames.PENALTYMARK.value,
    ]:
        if object_name not in preds or (
            ball_status_only_seen
            and object_name == u_dataset.CategoryNames.BALL.value
            and preds[object_name]["status"] == 2
        ):  # status == 2 is "guessed"
            pred_coords = []
        else:
            if isinstance(preds[object_name], list):
                pred_coords = extract_coordinates(preds[object_name])
            else:
                pred_coords = [[preds[object_name]["x"], preds[object_name]["y"]]]
        if object_name not in gt_frame:
            gt_coords = []
        else:
            if isinstance(gt_frame[object_name], list):
                gt_coords = extract_coordinates(gt_frame[object_name])
            else:
                gt_coords = [[gt_frame[object_name]["x"], gt_frame[object_name]["y"]]]
    else:
        raise ValueError("Invalid object_name.")

    # Convert coordinates to tensors
    pred_tensor = tf.constant(pred_coords, dtype=tf.float32)  # (N, 2)
    gt_tensor = tf.constant(gt_coords, dtype=tf.float32)  # (N, 2)

    # Match keypoints and return metrics
    return u_metrics.match_keypoints_image(pred_tensor, gt_tensor, threshold)

In [ ]:
def compare_predictions(
    groundtruth,
    model_preds,
    bhuman_preds,
    object_name,
    max_distance,
    threshold,
    ball_status_only_seen: bool = None,
) -> dict:
    if object_name == u_dataset.CategoryNames.INTERSECTIONS.value:
        metrics = {
            "model_true_positives": {k.name: 0 for k in u_labels.IntersectionType},
            "model_false_negatives": {k.name: 0 for k in u_labels.IntersectionType},
            "model_false_positives": {k.name: 0 for k in u_labels.IntersectionType},
            "bhuman_true_positives": {k.name: 0 for k in u_labels.IntersectionType},
            "bhuman_false_negatives": {k.name: 0 for k in u_labels.IntersectionType},
            "bhuman_false_positives": {k.name: 0 for k in u_labels.IntersectionType},
        }
        tp_matches = {
            "model": {k.name: [] for k in u_labels.IntersectionType},
            "bhuman": {k.name: [] for k in u_labels.IntersectionType},
        }
        intersection_types = [t.name for t in list(u_labels.IntersectionType)]

    else:
        metrics = {
            "model_true_positives": 0,
            "model_false_negatives": 0,
            "model_false_positives": 0,
            "bhuman_true_positives": 0,
            "bhuman_false_negatives": 0,
            "bhuman_false_positives": 0,
        }
        tp_matches = {"model": [], "bhuman": []}
        intersection_types = [None]

    # Iterate over each frame
    for idx, gt_frame in enumerate(groundtruth):
        frame_time = gt_frame["frame_time"]

        assert frame_time == model_preds[idx]["frame_time"]
        assert frame_time == bhuman_preds[idx]["frame_time"]

        camera = u_dataset_io.camera_from_label(gt_frame)
        intrinsics = u_dataset_io.intrinsics_from_label(gt_frame)

        if (
            object_name == u_dataset.CategoryNames.INTERSECTIONS.value
            and gt_frame[object_name]["ignore_sample"]
        ):
            continue

        for intersection_type in intersection_types:
            model_matches = process_object_metrics(
                model_preds[idx],
                gt_frame,
                object_name,
                threshold,
                intersection_type,
            )
            bhuman_matches = process_object_metrics(
                bhuman_preds[idx],
                gt_frame,
                object_name,
                threshold,
                intersection_type,
                ball_status_only_seen,
            )

            for prefix, matches in [("model", model_matches), ("bhuman", bhuman_matches)]:
                # Filter and get lengths for false positives, false negatives, and matched ground truth
                fp_len = tf.shape(
                    filter_coords_by_distance(
                        matches["fp_tensor"], camera, intrinsics, max_distance
                    )
                )[0]
                fn_len = tf.shape(
                    filter_coords_by_distance(
                        matches["fn_tensor"], camera, intrinsics, max_distance
                    )
                )[0]

                filtered_matches = filter_coords_by_distance(
                    matches["matches"],
                    camera,
                    intrinsics,
                    max_distance,
                )
                matches_len = tf.shape(filtered_matches)[0]

                if object_name == u_dataset.CategoryNames.INTERSECTIONS.value:
                    tp_matches[prefix][intersection_type].append(filtered_matches)

                    # Update metrics
                    metrics[f"{prefix}_true_positives"][intersection_type] += int(matches_len)
                    metrics[f"{prefix}_false_negatives"][intersection_type] += int(fn_len)
                    metrics[f"{prefix}_false_positives"][intersection_type] += int(fp_len)
                else:
                    tp_matches[prefix].append(filtered_matches)

                    # Update metrics
                    metrics[f"{prefix}_true_positives"] += int(matches_len)
                    metrics[f"{prefix}_false_negatives"] += int(fn_len)
                    metrics[f"{prefix}_false_positives"] += int(fp_len)

    # Save matches in .npy file
    status_str = ""
    if ball_status_only_seen is not None:
        status_str = "_seen" if ball_status_only_seen else "_seen+guessed"

    for key, value in tp_matches.items():
        for intersection_type in intersection_types:
            matches_list = value if isinstance(value, list) else value[intersection_type]
            model_tp_matches_concat = tf.concat(matches_list, axis=0)  # (B, 2, 2)
            np.save(
                Path(
                    save_path_for_matches,
                    f"{key}_{object_name}{status_str}{f'_{intersection_type}' if intersection_type is not None else ''}",
                ),
                model_tp_matches_concat.numpy(),
            )

    return metrics

In [ ]:
metrics_ball = compare_predictions(
    test_groundtruth,
    test_ball_model,
    test_bhuman,
    u_dataset.CategoryNames.BALL.value,
    max_distance=config["categories"][u_dataset.CategoryNames.BALL.value]["max_distance"],
    threshold=30.0,
    ball_status_only_seen=True,
)
print("==== Ball, status: seen ====")
print("=== Model ===")
print("TP: ", metrics_ball["model_true_positives"])
print("FP: ", metrics_ball["model_false_positives"])
print("FN: ", metrics_ball["model_false_negatives"])

print("=== B-Human ===")
print("TP : ", metrics_ball["bhuman_true_positives"])
print("FP: ", metrics_ball["bhuman_false_positives"])
print("FN: ", metrics_ball["bhuman_false_negatives"])

In [ ]:
metrics_ball = compare_predictions(
    test_groundtruth,
    test_ball_model,
    test_bhuman,
    u_dataset.CategoryNames.BALL.value,
    max_distance=config["categories"][u_dataset.CategoryNames.BALL.value]["max_distance"],
    threshold=30.0,
    ball_status_only_seen=False,
)
print("==== Ball, status: seen + guessed ====")
print("=== Model ===")
print("TP: ", metrics_ball["model_true_positives"])
print("FP: ", metrics_ball["model_false_positives"])
print("FN: ", metrics_ball["model_false_negatives"])

print("=== B-Human ===")
print("TP : ", metrics_ball["bhuman_true_positives"])
print("FP: ", metrics_ball["bhuman_false_positives"])
print("FN: ", metrics_ball["bhuman_false_negatives"])

In [ ]:
metrics_penaltyMark = compare_predictions(
    test_groundtruth,
    test_penaltymark_model,
    test_bhuman,
    u_dataset.CategoryNames.PENALTYMARK.value,
    max_distance=config["categories"][u_dataset.CategoryNames.PENALTYMARK.value]["max_distance"],
    threshold=30.0,
)

print("==== PenaltyMark ====")
print("=== Model ===")
print("TP: ", metrics_penaltyMark["model_true_positives"])
print("FP: ", metrics_penaltyMark["model_false_positives"])
print("FN: ", metrics_penaltyMark["model_false_negatives"])

print("=== B-Human ===")
print("TP : ", metrics_penaltyMark["bhuman_true_positives"])
print("FP: ", metrics_penaltyMark["bhuman_false_positives"])
print("FN: ", metrics_penaltyMark["bhuman_false_negatives"])

In [ ]:
metrics_intersections = compare_predictions(
    test_groundtruth,
    test_intersections_model,
    test_bhuman,
    u_dataset.CategoryNames.INTERSECTIONS.value,
    max_distance=config["categories"][u_dataset.CategoryNames.INTERSECTIONS.value]["max_distance"],
    threshold=30.0,
)

print("==== Intersections ====")
print("=== Model ===")
print("TP: ", metrics_intersections["model_true_positives"])
print("FP: ", metrics_intersections["model_false_positives"])
print("FN: ", metrics_intersections["model_false_negatives"])

print("=== B-Human ===")
print("TP : ", metrics_intersections["bhuman_true_positives"])
print("FP: ", metrics_intersections["bhuman_false_positives"])
print("FN: ", metrics_intersections["bhuman_false_negatives"])

#### Confusion Matrix
